# basic

In [4]:
%load_ext autoreload
%autoreload all

In [17]:
import polars as pl
import numpy as np
import pickle
import itertools
import networkx as nx

import src.configs_mapped as configs_mapped
from src.graph_fct import build_relations_graph, get_subgraphs_from_nodes
import src.propagate as propagate

# from src.tokenizer import GraphTokenizer
# from src.iterative_selection import iterative_approach_w_graph_red, IterativeMarginalGainMax

# prepare graph

In [6]:
df_relation = pl.read_parquet(f"{configs_mapped.GraphConfig().relation_path}")
df_concept_path = pl.read_parquet(f"{configs_mapped.GraphConfig().concept_path}")


In [7]:
# 1. get mapped concepts
mapped_ids = df_concept_path.filter(pl.col("is_mapped"))["id"].to_list()
len(mapped_ids)

46150

In [8]:
# 0. build 3 hops subgraph from mapped concepts, regardless type of relations
G = build_relations_graph(df_relation)
subgraphs = get_subgraphs_from_nodes(G, mapped_ids, max_distance=configs_mapped.TokenizerParam().max_dist_candidate)
combined = nx.compose_all([sg for sg in subgraphs.values() if sg is not None])

# 1. build subgraph relation table
df_relations_subgraph = pl.DataFrame(
    [(u, v, d["relation"]) for u, v, d in combined.edges(data=True)],
    schema=["src.id", "dst.id", "relation"],
    orient="row",
)
df_direct = df_relations_subgraph.with_columns(distance = 1).unique()
# df_self = (df_relations_subgraph
#            .select("src.id")
#            .with_columns(distance = 0,
#                            relation = pl.lit("IS_A"))
#              .with_columns(pl.col("src.id").alias("dst.id"))
#              .select(df_direct.columns)
#              .unique())
# mapped_rel_final = pl.concat([df_direct, df_self])

In [9]:
df_direct

src.id,dst.id,relation,distance
str,str,str,i32
"""29689003""","""95880003""","""IS_A""",1
"""117033003:116686009=168139001,…","""410607006""","""COMPONENT""",1
"""737669009""","""11555005""","""IS_A""",1
"""707606004""","""18718003""","""IS_A""",1
"""244436004""","""128319008""","""IS_A""",1
…,…,…,…
"""117033003:116686009=258520000,…","""129265001""","""METHOD""",1
"""117033003:116686009=472903000,…","""129265001""","""METHOD""",1
"""241655007""","""70258002""","""PROCEDURE_SITE_DIRECT""",1


# prepare input for propagation and sem coverage

In [61]:
# 1. find concept propagation depart
cpt_sources = list(set(df_direct["dst.id"].to_list())-set(df_direct["src.id"].to_list()))

# 2. build the edges
edges = []
for row in df_direct.iter_rows(named=True):
    u = row["src.id"]
    v = row["dst.id"]
    edges.append((u,v))

# pick a candidate set and flag
# candidate_test = pl.read_parquet(f"{configs_mapped.IterativeMarginalGain().path}_inv.parquet")["candidate_id"].to_list()
random_k = pl.read_parquet(f"{configs_mapped.Baselines().path}k_random_all_samples.parquet")

import numpy as np

Ks = np.arange(500, 12000, 500)

for k in Ks:
    candidate_test = random_k.filter((pl.col("k") == k) & (pl.col("iter") == 0))["candidate_id"]

    flags = dict()
    for cpt_source in cpt_sources:
        flags[cpt_source] = 0 # by default setting becasue they shouls have a value
    for candidate in candidate_test: 
        flags[candidate] = 1 # flag candidates to be 1


    result = propagate.propagate_n_get_value(edges, flags)
    df_result = pl.DataFrame({"concept": list(result.keys()), "sem_cov": [float(v) for v in result.values()]})
    cov = propagate.get_sem_cov_score(result, mapped_ids)
    print(f"Number of selected condidate: {k}, sem_cov score: {cov['sem_cov'].mean()}")
    print(len(cov), "of", len(mapped_ids), "mapped concepts resolved")

Number of selected condidate: 500, sem_cov score: 0.19015048879714275
46150 of 46150 mapped concepts resolved
Number of selected condidate: 1000, sem_cov score: 0.043905799258133975
46150 of 46150 mapped concepts resolved
Number of selected condidate: 1500, sem_cov score: 0.06214911657966389
46150 of 46150 mapped concepts resolved
Number of selected condidate: 2000, sem_cov score: 0.10661245579406053
46150 of 46150 mapped concepts resolved
Number of selected condidate: 2500, sem_cov score: 0.3428139925109336
46150 of 46150 mapped concepts resolved
Number of selected condidate: 3000, sem_cov score: 0.14607590723528868
46150 of 46150 mapped concepts resolved
Number of selected condidate: 3500, sem_cov score: 0.21094191134230406
46150 of 46150 mapped concepts resolved
Number of selected condidate: 4000, sem_cov score: 0.241081897321176
46150 of 46150 mapped concepts resolved
Number of selected condidate: 4500, sem_cov score: 0.30422548715391623
46150 of 46150 mapped concepts resolved
Numb

In [60]:
pl.read_parquet(f"{configs_mapped.Baselines().path}k_random_all_samples.parquet").filter((pl.col("k") == k)& (pl.col("iter") == 0))["candidate_id"]


candidate_id
str
"""121932001:116686009=445364004"""
"""473449006"""
"""5880005:{363702006=1263452006}"""
"""364706004"""
"""442095009"""
…
"""25884002+45764000:116686009=11…"
"""41170006:116686009=438660002,{…"
"""5407000"""
